<a href="https://colab.research.google.com/github/Aljwharah-h/SDAIA-AI-Agents-System-program/blob/main/semantic_search_assignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Set up

In [ ]:
%pip install langchain-community pypdf

Loading documents

In [ ]:
from langchain_core.documents
import Document from langchain_community.document_loaders
import PyPDFLoader
file_path = "https://files.eric.ed.gov/fulltext/ED536788.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()
print(len(docs))

In [ ]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0])

Splitting

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
 )
all_splits = text_splitter.split_documents(docs)
print(len(all_splits))

Embeddings

In [ ]:
%pip install langchain-huggingface sentence-transformers

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [ ]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)
assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Vector stores

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
vector_store = InMemoryVectorStore(embeddings)

In [ ]:
ids = vector_store.add_documents(all_splits)
print(ids)

In [ ]:
results = vector_store.similarity_search(
    "What does this document talk about??" )
print(results[0])

In [ ]:
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.
results = vector_store.similarity_search_with_score("How does the Transformer encoder work?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

In [ ]:
embedding = embeddings.embed_query("What are the advantages of the Transformer over RNNs?")
results = vector_store.similarity_search_by_vector(embedding)
print(results[0])

Retrievers

In [ ]:
retriever = vector_store.as_retriever(
     search_type="similarity",
     search_kwargs={"k": 1},
  )
retriever.batch(
    [
        "What is the main idea proposed in the pdf?",
        "What is data analysis?",
    ],
  )